In [1]:
!pip install vitaldb scikit-learn imbalanced-learn tensorflow pandas numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create folder for VitalDB files
import os
drive_path = "/content/drive/MyDrive/vitaldb_cases"
os.makedirs(drive_path, exist_ok=True)


Mounted at /content/drive


In [3]:
# ============================================================
# EARLY HYPOXIA PREDICTION PIPELINE
# Robust VitalDB version (Capstone + Publication Ready)
# ============================================================

"""
PIPELINE OVERVIEW (Diagram for Report / Slides)

VitalDB
   │
   ▼
Signal Cleaning & Interpolation
   │
   ▼
Rolling Feature Extraction
(mean / std / slope)
   │
   ▼
Leakage-Safe Labeling (future hypoxia)
   │
   ▼
Model Training (GroupKFold)
   │
   ├── Logistic Regression (Baseline)
   └── Random Forest (Main)
   │
   ▼
Risk Score (0–1)
   │
   ▼
Early Warning Alert (Bedside Use)
"""

# ============================================================
# IMPORTS
# ============================================================

import vitaldb
import pandas as pd
import numpy as np
from scipy.stats import linregress
import warnings

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


warnings.filterwarnings("ignore")

# ============================================================
# 1. DATA LOADING (ROBUST, OPTIONAL SIGNALS)
# ============================================================

def load_clean_vital_data(caseid):
    tracks = [
        'Solar8000/PLETH_SPO2',
        'Solar8000/HR',
        'Solar8000/RR',
        'Solar8000/ETCO2'
    ]

    try:
        data = vitaldb.load_case(caseid, tracks, interval=1)
        df = pd.DataFrame(data, columns=tracks)

        # SpO2 is mandatory
        df = df.dropna(subset=['Solar8000/PLETH_SPO2'])

        # Physiological sanity bounds
        df['Solar8000/PLETH_SPO2'] = df['Solar8000/PLETH_SPO2'].clip(50, 100)
        df['Solar8000/HR'] = df['Solar8000/HR'].clip(30, 220)

        # Optional signals
        for col in ['Solar8000/HR', 'Solar8000/RR', 'Solar8000/ETCO2']:
            df[col] = df[col].interpolate(limit=10)

        return df

    except Exception:
        return None


# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================

def calculate_slope(x):
    if len(x) < 2:
        return 0.0
    return linregress(range(len(x)), x)[0]


def create_features_and_labels(
    df,
    lookback_sec=60,
    lead_time_sec=120,
    hypoxia_thresh=94,
    sustain_sec=5
):
    # ---------- FEATURES ----------
    rolling = df.rolling(window=lookback_sec, min_periods=30)

    means = rolling.mean().add_suffix('_mean')
    stds = rolling.std().add_suffix('_std')
    slopes = rolling.apply(calculate_slope).add_suffix('_slope')

    X = pd.concat([means, stds, slopes], axis=1)

    # ---------- LABELS (LEAKAGE-SAFE) ----------
    hypoxic = (df['Solar8000/PLETH_SPO2'] < hypoxia_thresh).astype(int)

    sustained = hypoxic.rolling(
        window=sustain_sec,
        min_periods=sustain_sec
    ).min()

    future_event = (
        sustained.shift(-lead_time_sec)
        .rolling(window=lead_time_sec, min_periods=1)
        .max()
    )

    y = (future_event == 1).astype(int)

    # ---------- ALIGN ----------
    data = X.copy()
    data['label'] = y
    data = data.dropna(subset=['label'])

    # Subsample every 10 seconds
    data = data.iloc[::10]

    if data.empty:
        return None, None

    return data.drop(columns='label'), data['label']


# ============================================================
# 3. DATASET BUILDER
# ============================================================

def build_dataset(case_ids):
    X_all, y_all, groups = [], [], []

    print("\n📊 BUILDING DATASET\n")

    for cid in case_ids:
        raw = load_clean_vital_data(cid)

        if raw is None or len(raw) < 300:
            continue

        X, y = create_features_and_labels(raw)

        if X is None or len(y) < 10:
            continue

        print(f"Case {cid:4d} | samples={len(y):4d} | positives={y.sum():3d}")

        X_all.append(X)
        y_all.append(y)
        groups.append(np.full(len(y), cid))

    return (
        pd.concat(X_all),
        pd.concat(y_all),
        np.concatenate(groups)
    )


# ============================================================
# 4. EVENT-BASED EVALUATION (PUBLISHABLE)
# ============================================================

def event_based_metrics(y_true, y_prob, threshold=0.5, min_gap=30):
    """
    Early-warning evaluation:
    Did the model alert BEFORE hypoxia events?
    """
    alerts = (y_prob >= threshold).astype(int)

    events = []
    in_event = False
    for i, v in enumerate(y_true):
        if v == 1 and not in_event:
            events.append(i)
            in_event = True
        elif v == 0:
            in_event = False

    detected = 0
    lead_times = []

    for e in events:
        window = alerts[max(0, e - min_gap):e]
        if window.sum() > 0:
            detected += 1
            lead_times.append(e - np.where(window == 1)[0][-1])

    sensitivity = detected / len(events) if events else 0.0
    mean_lead = np.mean(lead_times) if lead_times else 0.0

    return sensitivity, mean_lead


def sensitivity_at_alert_rate(y_true, y_prob, alerts_per_hour):
    total_hours = len(y_true) / 360
    thresh = np.percentile(y_prob, 100 * (1 - alerts_per_hour * total_hours / len(y_prob)))
    sens, _ = event_based_metrics(y_true, y_prob, threshold=thresh)
    return sens


# ============================================================
# 5. MODEL TRAINING (BASELINE + MAIN)
# ============================================================

def run_pipeline(case_ids):
    X, y, groups = build_dataset(case_ids)

    print("\nLabel distribution:")
    print(y.value_counts())
    print("\nNOTE ON METRICS:")
    print("- Hypoxia is rare (<1–3%)")
    print("- Random AUPRC ≈ event rate")
    print("- Relative improvement matters more than absolute value\n")

    gkf = GroupKFold(n_splits=3)

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        print(f"\n🔁 Fold {fold}")

        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        # ====================================================
        # BASELINE: LOGISTIC REGRESSION (WITH IMPUTATION)
        # ====================================================

        lr_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                solver="liblinear"
            ))
        ])

        lr_pipeline.fit(X_tr, y_tr)
        lr_probs = lr_pipeline.predict_proba(X_te)[:, 1]

        print(
            f"LogReg | AUROC={roc_auc_score(y_te, lr_probs):.3f} "
            f"| AUPRC={average_precision_score(y_te, lr_probs):.3f}"
        )

        # ====================================================
        # MAIN MODEL: RANDOM FOREST (WITH IMPUTATION)
        # ====================================================

        rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )

        rf_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("rf", rf)
        ])

        model = CalibratedClassifierCV(
            rf_pipeline,
            cv=3,
            method='sigmoid'
        )

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_te)[:, 1]

        print(
            f"RF     | AUROC={roc_auc_score(y_te, probs):.3f} "
            f"| AUPRC={average_precision_score(y_te, probs):.3f}"
        )

        # ====================================================
        # EVENT-BASED EVALUATION
        # ====================================================

        sens, lead = event_based_metrics(y_te.values, probs)
        print(f"Event sensitivity={sens:.2f} | mean lead={lead:.1f} samples")

        print(
            f"Sensitivity @1 alert/hr: "
            f"{sensitivity_at_alert_rate(y_te.values, probs, 1):.2f}"
        )

    return model



# ============================================================
# 6. LIMITATIONS & CONTRIBUTION (FOR REPORT)
# ============================================================

"""
LIMITATIONS:
- Single dataset (VitalDB only)
- Extreme class imbalance
- No external / prospective validation
- Alert thresholds not clinically optimized

FUTURE WORK:
- Multi-center validation
- Clinician-tuned alert thresholds
- Alert burden optimization
- LightGBM comparison under same protocol

CONTRIBUTION:
This work does not propose a novel algorithm.
Instead, it presents a robust, leakage-safe,
deployment-aware framework for early hypoxia
prediction under extreme class imbalance.
"""

# ============================================================
# 7. EXECUTION
# ============================================================

if __name__ == "__main__":

    case_list = vitaldb.find_cases(
        ['Solar8000/PLETH_SPO2', 'Solar8000/HR']
    )[:300]

    model = run_pipeline(case_list)



📊 BUILDING DATASET

Case    1 | samples= 550 | positives= 14
Case    2 | samples= 758 | positives= 14
Case    3 | samples= 203 | positives= 25
Case    4 | samples=1019 | positives= 12
Case    5 | samples=1025 | positives=  0
Case    6 | samples= 233 | positives= 17
Case    7 | samples= 755 | positives= 14
Case    8 | samples= 271 | positives= 15
Case    9 | samples= 197 | positives= 16
Case   10 | samples=1020 | positives=  0
Case   11 | samples= 236 | positives= 13
Case   12 | samples=1548 | positives= 25
Case   13 | samples= 518 | positives= 21
Case   14 | samples= 188 | positives= 24
Case   15 | samples= 173 | positives= 39
Case   16 | samples= 621 | positives= 21
Case   17 | samples= 989 | positives= 12
Case   18 | samples= 200 | positives= 29
Case   19 | samples=1349 | positives=  0
Case   20 | samples=1283 | positives= 70
Case   21 | samples= 595 | positives= 14
Case   22 | samples= 700 | positives= 24
Case   23 | samples= 129 | positives= 28
Case   24 | samples= 299 | positives

Gradio

In [4]:
import joblib
from google.colab import files

# Save the trained calibrated RF model
joblib.dump(model, "hypoxia_rf_model.pkl")

print("✅ Model saved as hypoxia_rf_model.pkl")


files.download("hypoxia_rf_model.pkl")


✅ Model saved as hypoxia_rf_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
%%writefile app_gradio.py
import gradio as gr
import pandas as pd
import numpy as np
import joblib
import vitaldb
import plotly.graph_objects as go
from scipy.stats import linregress
import threading
import time

# Load model
try:
    model = joblib.load("hypoxia_rf_model.pkl")
except:
    model = None
    print("Warning: Model file not found. Prediction logic will fail.")

# Global controls
stop_monitoring = threading.Event()
is_paused = False  # New global flag for pause state

# --- UI LOGIC ---

def get_status_ui(risk, paused=False):
    """Returns HTML banner. Added 'paused' visual state."""
    if paused:
        color, text, audio_html = "#555555", "⏸️ MONITORING PAUSED", ""
    elif risk > 0.5:
        color, text = "#FF4444", "🚨 CRITICAL: HIGH HYPOXIA RISK"
        audio_html = '<audio autoplay loop><source src="https://actions.google.com/sounds/v1/alarms/beep_short.ogg" type="audio/ogg"></audio>'
    elif risk > 0.2:
        color, text, audio_html = "#FFBB33", "⚠️ WARNING: ELEVATED RISK", ""
    else:
        color, text, audio_html = "#00C851", "✅ STABLE", ""

    return f"""
    <div style="background-color: {color}; padding: 30px; border-radius: 15px; text-align: center; transition: all 0.5s ease;">
        <h1 style="color: white; margin: 0; font-family: sans-serif; letter-spacing: 2px;">{text}</h1>
        {audio_html}
    </div>
    """

# --- DATA PROCESSING ---

def calculate_slope(x):
    if len(x) < 2: return 0.0
    try: return linregress(range(len(x)), x)[0]
    except: return 0.0

def extract_features(df, window_size=60):
    if len(df) < window_size: return None
    window = df.iloc[-window_size:]
    tracks = ['Solar8000/PLETH_SPO2', 'Solar8000/HR', 'Solar8000/RR', 'Solar8000/ETCO2']
    feature_dict = {}
    for track in tracks:
        series = window[track].interpolate(limit=10).fillna(0) if track in window.columns else pd.Series([0]*60)
        feature_dict[f"{track}_mean"] = series.mean()
        feature_dict[f"{track}_std"] = series.std()
        feature_dict[f"{track}_slope"] = calculate_slope(series.values)
    expected_cols = [f"{t}_mean" for t in tracks] + [f"{t}_std" for t in tracks] + [f"{t}_slope" for t in tracks]
    return pd.DataFrame([feature_dict], columns=expected_cols)

def load_case_data(caseid):
    tracks = ['Solar8000/PLETH_SPO2', 'Solar8000/HR', 'Solar8000/RR', 'Solar8000/ETCO2']
    try:
        data = vitaldb.load_case(int(caseid), tracks, interval=1)
        df = pd.DataFrame(data, columns=tracks).dropna(subset=['Solar8000/PLETH_SPO2'])
        return df.interpolate(limit=10)
    except: return None

def create_plot(h_spo2, h_risk):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=h_spo2, name="SpO₂ (%)", line=dict(color="#00D9FF", width=3)))
    fig.add_trace(go.Scatter(y=h_risk, name="Risk (%)", yaxis="y2", line=dict(color="#FF4444", width=2, dash="dot")))
    fig.update_layout(
        template="plotly_dark", height=400, margin=dict(l=10, r=10, t=10, b=10),
        yaxis=dict(title="SpO₂", range=[80, 100]),
        yaxis2=dict(title="Risk Probability", overlaying="y", side="right", range=[0, 100]),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    return fig

# --- MONITORING LOOP ---

def toggle_pause():
    global is_paused
    is_paused = not is_paused
    return "▶️ Resume" if is_paused else "⏸️ Pause"

def stop_fn():
    global is_paused
    is_paused = False
    stop_monitoring.set()
    return get_status_ui(0), "⏸️ Pause"

def run_monitoring(case_id, jump_to_event, progress=gr.Progress()):
    global is_paused
    is_paused = False
    stop_monitoring.clear()

    data = load_case_data(case_id)
    if data is None:
        yield get_status_ui(0), None, "Error loading data", "--"
        return

    t_start = 60
    if jump_to_event:
        mask = data["Solar8000/PLETH_SPO2"] < 94
        if mask.any(): t_start = max(60, mask.idxmax() - 120)

    h_spo2, h_risk = [], []

    # Use index-based loop to allow pausing without losing place
    current_idx = t_start
    while current_idx < len(data):
        if stop_monitoring.is_set(): break

        # Pause logic: stay here while is_paused is True
        if is_paused:
            yield get_status_ui(0, paused=True), gr.skip(), gr.skip(), gr.skip()
            time.sleep(0.5)
            continue

        feat = extract_features(data.iloc[:current_idx+1], 60)
        if feat is None:
            current_idx += 2
            continue

        risk_prob = model.predict_proba(feat)[0][1] if model else 0.0
        current_spo2 = data.iloc[current_idx]["Solar8000/PLETH_SPO2"]

        h_spo2.append(current_spo2)
        h_risk.append(risk_prob * 100)
        h_spo2 = h_spo2[-100:]
        h_risk = h_risk[-100:]

        status_html = get_status_ui(risk_prob)
        fig = create_plot(h_spo2, h_risk)

        yield (
            status_html,
            fig,
            f"{current_spo2:.1f}%",
            f"{risk_prob*100:.1f}%"
        )

        current_idx += 2
        time.sleep(0.1)

# --- GRADIO INTERFACE ---

with gr.Blocks(theme=gr.themes.Default(primary_hue="blue"), title="AI Hypoxia Warning") as demo:
    gr.Markdown("# 🏥 Peri-operative Hypoxia Early Warning System")

    status_beacon = gr.HTML(get_status_ui(0))

    with gr.Row():
        with gr.Column():
            case_id_input = gr.Number(label="VitalDB Case ID", value=75, precision=0)
            jump_chk = gr.Checkbox(label="Jump to Hypoxia Event", value=True)
        with gr.Column():
            start_btn = gr.Button("▶️ START MONITORING", variant="primary")
            with gr.Row():
                pause_btn = gr.Button("⏸️ Pause", variant="secondary")
                stop_btn = gr.Button("⏹️ STOP SYSTEM", variant="stop")

    with gr.Row():
        spo2_val = gr.Label(label="Current SpO₂")
        risk_val = gr.Label(label="AI Predicted Risk")

    plot_view = gr.Plot(label="Live Vital Sign Trends")

    start_btn.click(
        fn=run_monitoring,
        inputs=[case_id_input, jump_chk],
        outputs=[status_beacon, plot_view, spo2_val, risk_val]
    )

    # New button functionality
    pause_btn.click(fn=toggle_pause, outputs=[pause_btn])
    stop_btn.click(fn=stop_fn, outputs=[status_beacon, pause_btn])

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Writing app_gradio.py


In [6]:
pip install gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 70.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [ ]:
!pip install gradio -q

# Kill any existing processes
!pkill -f streamlit
!pkill -f gradio

# Run the Gradio app
!python app_gradio.py

# Running on public URL: https://xxxxx.gradio.live

/content/app_gradio.py:152: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue="blue"), title="AI Hypoxia Warning") as demo:
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8b1339fb9ba619759b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
